# Phase 3-3: Detection Performance

Evaluate how well we can detect adversarial examples using feature distances.

In [ ]:
import sys
import os
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, precision_recall_curve

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

print(f"Colab: {IN_COLAB}")

In [ ]:
# paths
if IN_COLAB:
    BASE = "/content/drive/MyDrive/Colab Notebooks/data"
else:
    BASE = "/Users/tyreecruse/Desktop/CS230/Project/Data"

FEATURES_DIR = f"{BASE}/analysis/yolo_features"
RESULTS_DIR = f"{BASE}/analysis/results"

os.makedirs(RESULTS_DIR, exist_ok=True)

ATTACKS = [
    "fgsm_030", "fgsm_045", "fgsm_060", "fgsm_075", "fgsm_090", "fgsm_105",
    "gaussian_010", "gaussian_050", "gaussian_150", "gaussian_200", "gaussian_250",
    "patches",
]

In [ ]:
# load clean features
print("Loading clean features...")
with open(f"{FEATURES_DIR}/clean_yolo_features.pkl", 'rb') as f:
    cleanData = pickle.load(f)
cleanFeats = cleanData['features']
print(f"Shape: {cleanFeats.shape}")

# compute centroid
centroid = cleanFeats.mean(axis=0)
centroid = centroid / np.linalg.norm(centroid)

# clean distances
cleanDist = 1 - (cleanFeats @ centroid)
cleanMean = cleanDist.mean()
cleanStd = cleanDist.std()
print(f"Clean dist: mean={cleanMean:.4f}, std={cleanStd:.4f}")

In [ ]:
# compute detection metrics for each attack
allMetrics = []

for attackName in ATTACKS:
    print(f"\n{attackName}...", end=" ")
    
    try:
        # load adv features
        with open(f"{FEATURES_DIR}/{attackName}_yolo_features.pkl", 'rb') as f:
            advData = pickle.load(f)
        advFeats = advData['features']
        advDist = 1 - (advFeats @ centroid)
        
        # set up binary classification: clean=0, adv=1
        allDist = np.concatenate([cleanDist, advDist])
        labels = np.concatenate([np.zeros(len(cleanDist)), np.ones(len(advDist))])
        
        # ROC curve
        fpr, tpr, _ = roc_curve(labels, allDist)
        rocAuc = auc(fpr, tpr)
        
        # PR curve
        prec, rec, prThresh = precision_recall_curve(labels, allDist)
        prAuc = auc(rec, prec)
        
        # optimal F1
        f1s = 2 * (prec[:-1] * rec[:-1]) / (prec[:-1] + rec[:-1] + 1e-8)
        optF1 = f1s.max()
        
        # sigma-based thresholds (2, 3, 4 sigma)
        sigmaMetrics = {}
        for sigma in [2, 3, 4]:
            thresh = cleanMean + sigma * cleanStd
            preds = (allDist > thresh).astype(int)
            
            tp = ((preds == 1) & (labels == 1)).sum()
            fp = ((preds == 1) & (labels == 0)).sum()
            fn = ((preds == 0) & (labels == 1)).sum()
            tn = ((preds == 0) & (labels == 0)).sum()
            
            sigmaMetrics[f'{sigma}s_recall'] = tp / (tp + fn) if (tp + fn) > 0 else 0
            sigmaMetrics[f'{sigma}s_prec'] = tp / (tp + fp) if (tp + fp) > 0 else 0
            sigmaMetrics[f'{sigma}s_fpr'] = fp / (fp + tn) if (fp + tn) > 0 else 0
        
        # parse attack type
        if attackName.startswith('fgsm'):
            attackType = 'FGSM'
            strength = int(attackName.split('_')[1]) / 1000
        elif attackName.startswith('gaussian'):
            attackType = 'Gaussian'
            strength = int(attackName.split('_')[1]) / 1000
        else:
            attackType = 'Patch'
            strength = None
        
        metrics = {
            'attack': attackName,
            'attackType': attackType,
            'strength': strength,
            'rocAuc': rocAuc,
            'prAuc': prAuc,
            'optF1': optF1,
            'fpr': fpr,
            'tpr': tpr,
            'prPrec': prec,
            'prRec': rec,
            **sigmaMetrics,
        }
        allMetrics.append(metrics)
        
        print(f"AUC={rocAuc:.3f}")
        
    except FileNotFoundError:
        print("not found")
    except Exception as e:
        print(f"error: {e}")

print(f"\nComputed metrics for {len(allMetrics)} attacks")

In [ ]:
# display summary
summary = []
for m in allMetrics:
    summary.append({
        'Attack': m['attack'],
        'Type': m['attackType'],
        'Strength': m['strength'] if m['strength'] else '-',
        'ROC-AUC': round(m['rocAuc'], 3),
        'PR-AUC': round(m['prAuc'], 3),
        'F1': round(m['optF1'], 3),
        '3σ Recall': round(m['3s_recall'], 3),
        '3σ FPR': round(m['3s_fpr'], 4),
    })

df = pd.DataFrame(summary)

print("\n" + "="*80)
print("DETECTION PERFORMANCE")
print("="*80)
print(df.to_string(index=False))

# save
df.to_csv(f"{RESULTS_DIR}/detection_performance.csv", index=False)
print(f"\nSaved to {RESULTS_DIR}/detection_performance.csv")

In [ ]:
# plot ROC curves
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

colors = plt.cm.viridis(np.linspace(0, 1, 7))

# FGSM
fgsmMetrics = sorted([m for m in allMetrics if m['attackType'] == 'FGSM'], key=lambda x: x['strength'])
for i, m in enumerate(fgsmMetrics):
    axes[0].plot(m['fpr'], m['tpr'], label=f"{m['strength']:.3f} (AUC={m['rocAuc']:.2f})", color=colors[i], linewidth=2)
axes[0].plot([0,1], [0,1], 'k--', alpha=0.5)
axes[0].set_xlabel('FPR')
axes[0].set_ylabel('TPR')
axes[0].set_title('FGSM')
axes[0].legend(loc='lower right', fontsize=9)
axes[0].grid(alpha=0.3)

# Gaussian
gaussMetrics = sorted([m for m in allMetrics if m['attackType'] == 'Gaussian'], key=lambda x: x['strength'])
for i, m in enumerate(gaussMetrics):
    axes[1].plot(m['fpr'], m['tpr'], label=f"{m['strength']:.3f} (AUC={m['rocAuc']:.2f})", color=colors[i], linewidth=2)
axes[1].plot([0,1], [0,1], 'k--', alpha=0.5)
axes[1].set_xlabel('FPR')
axes[1].set_ylabel('TPR')
axes[1].set_title('Gaussian')
axes[1].legend(loc='lower right', fontsize=9)
axes[1].grid(alpha=0.3)

# Patches
patchMetrics = [m for m in allMetrics if m['attackType'] == 'Patch']
for m in patchMetrics:
    axes[2].plot(m['fpr'], m['tpr'], label=f"Patch (AUC={m['rocAuc']:.2f})", linewidth=2)
axes[2].plot([0,1], [0,1], 'k--', alpha=0.5)
axes[2].set_xlabel('FPR')
axes[2].set_ylabel('TPR')
axes[2].set_title('Patches')
axes[2].legend(loc='lower right')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/roc_curves.png", dpi=150)
plt.show()

In [ ]:
# AUC vs strength
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# FGSM
x = [m['strength'] for m in fgsmMetrics]
y = [m['rocAuc'] for m in fgsmMetrics]
axes[0].plot(x, y, 'ro-', linewidth=2, markersize=8)
axes[0].axhline(0.9, color='green', linestyle='--', alpha=0.5, label='AUC=0.9')
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Random')
axes[0].set_xlabel('FGSM ε')
axes[0].set_ylabel('ROC-AUC')
axes[0].set_title('Detection vs FGSM')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_ylim([0.4, 1.02])

# Gaussian
x = [m['strength'] for m in gaussMetrics]
y = [m['rocAuc'] for m in gaussMetrics]
axes[1].plot(x, y, 'go-', linewidth=2, markersize=8)
axes[1].axhline(0.9, color='green', linestyle='--', alpha=0.5, label='AUC=0.9')
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Random')
axes[1].set_xlabel('Gaussian σ')
axes[1].set_ylabel('ROC-AUC')
axes[1].set_title('Detection vs Gaussian')
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_ylim([0.4, 1.02])

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/auc_vs_strength.png", dpi=150)
plt.show()

In [ ]:
print("\n" + "="*60)
print("KEY FINDINGS")
print("="*60)

print("\nFGSM (3σ threshold):")
for m in fgsmMetrics:
    rec = m['3s_recall']
    status = "Reliable" if rec > 0.9 else "Moderate" if rec > 0.5 else "Poor"
    print(f"  ε={m['strength']:.3f}: Recall={rec:.0%}, FPR={m['3s_fpr']:.1%} - {status}")

print("\nGaussian (3σ threshold):")
for m in gaussMetrics:
    rec = m['3s_recall']
    status = "Reliable" if rec > 0.9 else "Moderate" if rec > 0.5 else "Poor"
    print(f"  σ={m['strength']:.3f}: Recall={rec:.0%}, FPR={m['3s_fpr']:.1%} - {status}")

if patchMetrics:
    m = patchMetrics[0]
    print(f"\nPatches: AUC={m['rocAuc']:.3f}, 3σ-Recall={m['3s_recall']:.0%}")
    if m['rocAuc'] < 0.7:
        print("  → Patches poorly detected with global features")